# GOIT pipelines summary sheets — June 2026 release (v2)

Produces summary tables (km by region, country, owner, start year) for the GOIT pipelines
data release. Writes a single Excel file that is pasted into the shared summary tables:
https://docs.google.com/spreadsheets/d/1OYH6D7c-D0FsL5GzBGijtkmvQCTkBUclj-UVoOieUFo/edit

Refactor of `GOIT-pipelines-summary-sheets-June2026-release.ipynb`, unchanged in the numbers
it produces. Differences:

- standard `pd` / `np` aliases, all imports at the top
- a single config block (spreadsheet key, fuel type, region, output dir)
- fuel buckets, status lists, and per-fuel labels live in one dict — no triple-if chains
- numeric columns are explicitly coerced at load time (`pd.to_numeric(..., errors="coerce")`),
  so empty-string sentinels can't slip past `.notna()` and crash `.quantile()` / `.sort()`
- region selection and country/region subsets in one cell so they can't be run out of order
- country filtering uses `.isin()` rather than regex `.contains('|'.join(...))` — no UserWarnings,
  no accidental substring matches (e.g. "Iran" inside "Iraq")
- owner/parent expansion builds one DataFrame from a list of dicts rather than growing it with
  `pd.concat` in a loop (O(n) vs O(n²))
- start-year table's year range is derived from the data rather than hard-coded
- output Excel file is written next to this notebook via `pathlib.Path`


## imports and configuration

In [ ]:
import datetime
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pygsheets
import pytz


In [ ]:
# === config ===========================================================
# Set FUEL_TYPE to one of: "Gas", "Oil", "NGL". Run the notebook once per fuel.
FUEL_TYPE = "Oil"

# Source spreadsheet (June 2026 GOIT oil/NGL release).
# Past keys, for reference:
#   1WaBMIdfRWqSqXUw7_cKXo3RipyhPdnNN8flqEYfMZIA  — Dec 2023 gas
#   1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek  — CURRENT (rolling)
#   1OXybaZOn0f2ONB6d_J0A3SG2bJ660C2Kr8fuc5o8cjs  — Dec 2024 GGIT
#   1xjaeq0OwdN-Orht7Q7ynPHB2uY_gnMvAeha-QHkzoZw  — Nov 2025 GGIT
SPREADSHEET_KEY = "1DrG5_zSa-PxLBaa1eBAOgONISgjeM7Te1El6SObiq3E"

# Where to write the Excel summary. Defaults to alongside this notebook.
OUTPUT_DIR = Path.cwd()

# Region filter:
#   "Global"             — every country
#   "AsiaGasTracker"     — countries flagged AsiaGasTracker == "Yes"
#   "EuroGasTracker"     — countries flagged EuroGasTracker == "Yes"
#   "AfricaGasTracker"   — countries flagged AfricaGasTracker == "Yes"
#   "LatinAmericaTracker" — countries flagged LatinAmericaTracker == "Yes"
REGION_NAME = "Global"


In [ ]:
# fuel buckets — union of oil + NGL must equal the "Oil-NGL" fuel_options list in
# GOIT-GGIT-data-requests/notebooks/convert-ggit-goit-to-tracker-release-downloads-CURRENT.ipynb.
# "Oil, NGL" and "Oil, NGL, naphtha" intentionally appear in BOTH oil and NGL buckets.
GAS_FUEL_OPTIONS = ["Gas", "Gas and Hydrogen"]
OIL_FUEL_OPTIONS = [
    "Oil",
    "Oil, NGL",
    "Oil, NGL, naphtha",
    "Oil products (only)",
    "Oil, oil products",
    "Oil, condensate",
    "CO2",
]
NGL_FUEL_OPTIONS = [
    "NGL",
    "NGL, oil products",
    "Oil, NGL",
    "Oil, NGL, naphtha",
    "LPG",
    "Naphtha (only)",
    "Naphtha, oil products",
    "Condensate",
    "Condensate/NGL",
]

FUEL_CONFIG = {
    "Gas": {"sheet": "Gas pipelines",     "options": GAS_FUEL_OPTIONS, "label": "Gas"},
    "Oil": {"sheet": "Oil/NGL pipelines", "options": OIL_FUEL_OPTIONS, "label": "Oil"},
    "NGL": {"sheet": "Oil/NGL pipelines", "options": NGL_FUEL_OPTIONS, "label": "NGL"},
}
assert FUEL_TYPE in FUEL_CONFIG, f"unknown FUEL_TYPE {FUEL_TYPE!r}"
FUEL_OPTIONS = FUEL_CONFIG[FUEL_TYPE]["options"]
FUEL_LABEL = FUEL_CONFIG[FUEL_TYPE]["label"]


In [ ]:
# status columns, in the order they appear in the output spreadsheet
STATUS_LIST = [
    "proposed",
    "construction",
    "shelved",
    "cancelled",
    "operating",
    "idle",
    "mothballed",
    "retired",
]
IN_DEV_COL = "in development (proposed + construction)"
EXCEL_STATUS_LIST = [
    "proposed",
    "construction",
    IN_DEV_COL,
    "shelved",
    "cancelled",
    "operating",
    "idle",
    "mothballed",
    "retired",
]

# Columns that should be numeric. pygsheets returns everything as strings, so we coerce
# at load time (a single source of truth) — this prevents the entire class of bug where
# an empty string slips past .notna() and crashes .quantile() / sort().
NUMERIC_COLS_PIPES = [
    "LengthMergedKm",
    "StartYearEarliest",
    "ProposalYear",
    "ConstructionYear",
    "ShelvedYear",
    "CancelledYear",
    "CostUSDPerKm",
    "CapacityBbl",
    "DiameterInches",
]
NUMERIC_COLS_RATIOS = [
    "LengthMergedKmByCountry",
    "LengthEstimateKmByCountry",  # may contain comma-formatted strings like "1,236.43"
    "LengthKnownKmByCountry",     # used by the cost-per-km estimate
    "LengthPerCountryFraction",
    "CostUSDPerKm",
]


## load the source workbook

In [ ]:
gc = pygsheets.authorize(service_account_env_var="GDRIVE_API_CREDENTIALS")
spreadsheet = gc.open_by_key(SPREADSHEET_KEY)


def load_sheet(title: str, start: str | None = None) -> pd.DataFrame:
    """Read a worksheet into a DataFrame. `start` is the top-left A1 cell when the
    sheet has header rows above the data (e.g. "A3" skips two prefix rows)."""
    ws = spreadsheet.worksheet("title", title)
    return ws.get_as_df(start=start) if start else ws.get_as_df()


def coerce_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Convert listed columns to numeric. Strips thousands separators (commas) first,
    then `pd.to_numeric(errors="coerce")` turns anything unparseable into NaN.
    Columns absent from `df` are silently skipped."""
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        s = df[col]
        if s.dtype == object:
            s = s.astype(str).str.replace(",", "", regex=False)
        df[col] = pd.to_numeric(s, errors="coerce")
    return df


pipes_sheet_title = FUEL_CONFIG[FUEL_TYPE]["sheet"]
pipes_df_orig = load_sheet(pipes_sheet_title, start="A3").drop(
    columns="WKTFormat", errors="ignore"
)
country_ratios_df = load_sheet("Country ratios by pipeline").drop(
    columns="WKTFormat", errors="ignore"
)
owners_df_orig = load_sheet("Pipeline operators/owners", start="A2")
parent_metadata_df = load_sheet("Parent metadata (3/3)", start="A3").set_index("Parent")
region_df_orig = load_sheet("Country dictionary", start="A2")

# coerce numeric columns at load time — single source of truth for type cleanup
pipes_df_orig = coerce_numeric(pipes_df_orig, NUMERIC_COLS_PIPES)
country_ratios_df = coerce_numeric(country_ratios_df, NUMERIC_COLS_RATIOS)


## clean inputs

In [ ]:
def clean_placeholders(df: pd.DataFrame) -> pd.DataFrame:
    """Replace `--` and empty-string sentinels with NaN."""
    return df.replace({"--": np.nan, "": np.nan})


# drop rows missing essential identifiers
pipes_df_orig = pipes_df_orig.loc[
    (pipes_df_orig["PipelineName"] != "") & (pipes_df_orig["Wiki"] != "")
].copy()

# subset pipes to the chosen fuel bucket; collapse "Gas and Hydrogen" -> "Gas" for gas runs
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig["Fuel"].isin(FUEL_OPTIONS)].copy()
if FUEL_TYPE == "Gas":
    pipes_df_orig.loc[pipes_df_orig["Fuel"] == "Gas and Hydrogen", "Fuel"] = "Gas"
    country_ratios_df.loc[country_ratios_df["Fuel"] == "Gas and Hydrogen", "Fuel"] = "Gas"

country_ratios_df = country_ratios_df.loc[country_ratios_df["Wiki"] != ""].copy()
country_ratios_df = clean_placeholders(country_ratios_df)

owners_df_orig = owners_df_orig.loc[
    (owners_df_orig["ProjectID"] != "")
    & (owners_df_orig["Wiki"] != "")
    & (owners_df_orig["Status"] != "N/A")
].set_index("ProjectID")
owners_df_orig = clean_placeholders(owners_df_orig)

pipes_df_orig = clean_placeholders(pipes_df_orig)


## region selection and country / region subsets

Selects countries for the chosen `REGION_NAME` and subsets `country_ratios_df` and `pipes_df_orig`
to those countries. Combined in one cell so they can't be run out of order.

In [ ]:
if REGION_NAME == "Global":
    region_df_touse = region_df_orig
else:
    if REGION_NAME not in region_df_orig.columns:
        raise KeyError(
            f"REGION_NAME={REGION_NAME!r} but region_df_orig has no column {REGION_NAME!r}; "
            f"available: {[c for c in region_df_orig.columns if c.endswith('Tracker')]}"
        )
    region_df_touse = region_df_orig.loc[region_df_orig[REGION_NAME] == "Yes"]

region_df_touse_cleaned = region_df_touse.loc[
    (region_df_touse["Region"] != "--") & (region_df_touse["SubRegion"] != "--")
]
multiindex_region_subregion = (
    region_df_touse_cleaned.groupby(["Region", "SubRegion"])["Country"].count().index
)

# isin() against an explicit set — no regex, no substring false-positives
countries_in_region = set(region_df_touse["Country"])

country_ratios_df_touse = country_ratios_df.loc[
    country_ratios_df["Country"].isin(countries_in_region)
].copy()
pipes_df_touse = pipes_df_orig.loc[
    pipes_df_orig["CountriesOrAreas"].isin(countries_in_region)
].copy()

print(f"region={REGION_NAME!r}: {len(countries_in_region)} countries, "
      f"{len(pipes_df_touse)} pipeline rows, {len(country_ratios_df_touse)} ratio rows")
multiindex_region_subregion


## set up Excel writer

In [ ]:
today = datetime.date.today().isoformat()
output_path = OUTPUT_DIR / f"GOIT-Summary-Sheets-{FUEL_LABEL}-{today}.xlsx"
print(f"writing to {output_path}")
excel_writer = pd.ExcelWriter(output_path)


## km by region and km by country

In [ ]:
country_ratios_df_subset = country_ratios_df_touse.loc[
    country_ratios_df_touse["Fuel"].isin(FUEL_OPTIONS)
]

country_list = sorted(country_ratios_df_subset["Country"].dropna().unique())


def pivot_km(group_cols, index) -> pd.DataFrame:
    """Sum LengthMergedKmByCountry by Status × group_cols, reshape to wide form."""
    grouped = (
        country_ratios_df_subset.groupby([*group_cols, "Status"])["LengthMergedKmByCountry"]
        .sum()
        .unstack("Status")
        .reindex(columns=STATUS_LIST)
        .reindex(index=index)
        .fillna(0)
    )
    grouped[IN_DEV_COL] = grouped[["proposed", "construction"]].sum(axis=1)
    return grouped[EXCEL_STATUS_LIST]


km_by_country = pivot_km(["Country"], country_list)
km_by_country.index.name = "Country"

km_by_region = pivot_km(["Region", "SubRegion"], multiindex_region_subregion)
km_by_region.index.names = ["Region", "Subregion"]

km_by_country.loc["Total"] = km_by_country.sum(axis=0).values
km_by_region.loc["Total"] = km_by_region.sum(axis=0).values

# drop countries with no km in any status, then blank out zeros for display
km_by_country = km_by_country.loc[~(km_by_country == 0).all(axis=1)]
km_by_country = km_by_country.replace(0, "")
km_by_region = km_by_region.replace(0, "")

km_by_region.to_excel(excel_writer, sheet_name="Kilometers by region")
km_by_country.to_excel(excel_writer, sheet_name="Kilometers by country")
km_by_region


## km by parent company

Each pipeline row has a `Parent` string like `"TC Energy Corp [60%]; Sempra Energy [40%]"`.
We parse this into per-owner rows, weight the per-country km by the ownership fraction, then
pivot to status columns.

In [ ]:
CJK_RE = re.compile(r"[\u4e00-\u9fff]+")
PERCENT_RE = re.compile(r"\d+(?:\.\d+)?%")
BRACKETS_RE = re.compile(r" \[.*?\]")


def parse_parent_string(parent_string) -> tuple[list[str], list[float]]:
    """Split a Parent cell into ([owner, ...], [fraction, ...]). Empty/NaN parent
    returns (["unknown"], [1.0]). If a percentage is missing, the leftover fraction is
    divided evenly across the owners without one."""
    if parent_string is None or (isinstance(parent_string, float) and np.isnan(parent_string)):
        return ["unknown"], [1.0]
    parent_string = str(parent_string).strip()
    if not parent_string:
        return ["unknown"], [1.0]

    # non-researched QCC owner: leading CJK character with a [100.00%] suffix; keep verbatim
    if CJK_RE.match(parent_string[:1]) and parent_string.endswith("[100.00%]"):
        return [parent_string.removesuffix(" [100.00%]")], [1.0]

    parents = BRACKETS_RE.sub("", parent_string).split("; ")
    pcts = [float(m.rstrip("%")) / 100.0 for m in PERCENT_RE.findall(parent_string)]

    if len(parents) != len(pcts):
        if not pcts:
            pcts = [1.0 / len(parents)] * len(parents)
        else:
            n_missing = len(parents) - len(pcts)
            leftover = 1.0 - float(np.nansum(pcts))
            pcts = pcts + [leftover / n_missing] * n_missing
    return parents, pcts


In [ ]:
# guard: every row in our subset must have a ProjectID
missing_pid = country_ratios_df_subset["ProjectID"].isna()
if missing_pid.any():
    raise ValueError(
        f"missing ProjectID in rows: {country_ratios_df_subset.index[missing_pid].tolist()}"
    )

# build owner-parent rows as a list of dicts, then one DataFrame at the end
# (avoids O(n²) repeated pd.concat in a loop)
records = []
for row in country_ratios_df_subset.itertuples(index=False):
    parents, pcts = parse_parent_string(row.Parent)
    for parent, frac in zip(parents, pcts):
        records.append(
            {
                "Parent": parent,
                "ProjectID": row.ProjectID,
                "FractionOwnership": frac,
                "Country": row.Country,
                "Status": row.Status,
                "LengthMergedKmByCountry": row.LengthMergedKmByCountry,
            }
        )

owner_parent_calculations_df = pd.DataFrame.from_records(records)
owner_parent_calculations_df["KmOwnership"] = (
    owner_parent_calculations_df["FractionOwnership"]
    * owner_parent_calculations_df["LengthMergedKmByCountry"]
)
owner_parent_calculations_df


In [ ]:
owners_km_by_status_df = (
    owner_parent_calculations_df.groupby(["Parent", "Status"])["KmOwnership"]
    .sum()
    .unstack("Status")
    .reindex(columns=STATUS_LIST)
)
owners_km_by_status_df[IN_DEV_COL] = owners_km_by_status_df[["proposed", "construction"]].sum(axis=1)
owners_km_by_status_df = owners_km_by_status_df[EXCEL_STATUS_LIST]

owners_km_by_status_df.loc["Total"] = owners_km_by_status_df.sum(axis=0, min_count=0).values

owners_km_by_status_df = owners_km_by_status_df.replace({np.nan: "", 0: ""})
owners_km_by_status_df.to_excel(excel_writer, sheet_name="Kilometers by owner")
owners_km_by_status_df


## km by start year, status

In [ ]:
def km_by_year(status_values, year_col: str) -> pd.Series:
    """Sum LengthMergedKm for pipes with the given status(es), grouped by `year_col`."""
    if isinstance(status_values, str):
        status_values = [status_values]
    subset = pipes_df_touse.loc[
        pipes_df_touse["Status"].isin(status_values)
        & pipes_df_touse["Fuel"].isin(FUEL_OPTIONS)
        & pipes_df_touse[year_col].notna()  # drop missing-year rows so they don't form a NaN group
    ]
    # year cols were coerced to float at load time; cast index back to int for clean display
    grouped = subset.groupby(year_col)["LengthMergedKm"].sum()
    grouped.index = grouped.index.astype(int)
    return grouped


pipes_started_sum      = km_by_year("operating",    "StartYearEarliest")
pipes_construction_sum = km_by_year("construction", "ConstructionYear")
pipes_proposed_sum     = km_by_year("proposed",     "ProposalYear")

# derive the year range from the data, with a floor at 1980 and ceiling at 2025
all_years = pd.concat([pipes_started_sum, pipes_construction_sum, pipes_proposed_sum]).index
year_min = int(min(1980, all_years.min())) if len(all_years) else 1980
year_max = int(max(2025, all_years.max())) if len(all_years) else 2025
year_index = pd.Index(range(year_min, year_max + 1), name="Start year")

km_by_start_year = pd.DataFrame(index=year_index)
km_by_start_year[f"{FUEL_LABEL} pipeline km operating"]    = pipes_started_sum
km_by_start_year[f"{FUEL_LABEL} pipeline km construction"] = pipes_construction_sum
km_by_start_year[f"{FUEL_LABEL} pipeline km proposed"]     = pipes_proposed_sum
km_by_start_year = km_by_start_year.fillna(0)
km_by_start_year.loc["Total"] = km_by_start_year.sum(axis=0)

km_by_start_year.to_excel(excel_writer, sheet_name="Kilometers by start year")


## cost estimates

Builds two artifacts. First, regional and subregional mean `CostUSDPerKm`, drawn from the
**global** set of pipelines (by country-fraction) that have both a known length and a known
per-km cost, trimmed to the [2.5%, 97.5%] interquantile range of `CostUSDPerKm` to drop
extreme outliers. The means are computed globally on purpose — narrowing to a single
`REGION_NAME` would leave too few datapoints in each subregion.

Sparse-sample fallbacks (extends the reference notebook):

- a region with fewer than `MIN_REGION_DATAPOINTS` unique pipelines uses the **global** mean
- a subregion with fewer than `MIN_SUBREGION_DATAPOINTS` unique pipelines uses its (possibly
  already-globalized) region's mean

Second, per-pipeline `CostUSDEstimate = LengthKnownKmByCountry × subregion_mean_cost_per_km`,
overwritten by the row's actual `LengthKnownKmByCountry × CostUSDPerKm` wherever both
populated. Aggregated into capex (USD billions) by status × country and status × region.

Reproduces the cost logic from `2025-q4-gas-pipelines/gas-pipelines-2025-research-calculations-for-costs.ipynb`,
adapted to the v2 fuel/region config and `pivot_*` helper style.

In [ ]:
# Cost-per-km region/subregion means are built from the full global fuel set so that
# small REGION_NAME filters don't shrink the sample size — only the final capex pivot is
# restricted to the chosen region.
country_ratios_fuel_df = country_ratios_df.loc[country_ratios_df["Fuel"].isin(FUEL_OPTIONS)]

cost_df = pipes_df_orig.loc[pipes_df_orig["CostUSDPerKm"].notna()]
if cost_df.empty:
    raise RuntimeError(
        f"no {FUEL_LABEL} pipelines have a numeric CostUSDPerKm — cost estimates cannot run"
    )

q_lo, q_hi = cost_df["CostUSDPerKm"].quantile([0.025, 0.975])
print(f"CostUSDPerKm 2.5% / 97.5% quantiles: {q_lo:,.0f} / {q_hi:,.0f}")

# country-ratio rows that contribute to the regional means: need both a known
# CostUSDPerKm and a known length, with cost inside the trimmed window
ratios_with_cost = country_ratios_fuel_df.loc[
    country_ratios_fuel_df["CostUSDPerKm"].notna()
    & country_ratios_fuel_df["LengthKnownKmByCountry"].notna()
    & country_ratios_fuel_df["CostUSDPerKm"].between(q_lo, q_hi, inclusive="neither")
]
print(f"country-ratio rows feeding the cost averages: {len(ratios_with_cost):,}")

# global region/subregion lists & subregion→region lookup come from the full country
# dictionary — the regional means are global; the REGION_NAME filter is applied later
region_dict_clean = region_df_orig.loc[
    (region_df_orig["Region"] != "--") & (region_df_orig["SubRegion"] != "--")
]
region_list_global = sorted(region_dict_clean["Region"].dropna().unique())
subregion_list_global = sorted(region_dict_clean["SubRegion"].dropna().unique())
dict_subregion_region = dict(
    zip(region_dict_clean["SubRegion"], region_dict_clean["Region"])
)


def cost_table(level_col: str, level_values: list[str]) -> pd.DataFrame:
    """Mean CostUSDPerKm and unique-ProjectID count, grouped by `level_col`
    (Region or SubRegion)."""
    grouped = ratios_with_cost.groupby(level_col)
    out = pd.DataFrame(
        index=pd.Index(level_values, name=level_col),
        columns=["CostUSDPerKm", "DataPoints"],
        dtype=float,
    )
    out["CostUSDPerKm"] = grouped["CostUSDPerKm"].mean()
    out["DataPoints"] = grouped["ProjectID"].nunique()
    out["DataPoints"] = out["DataPoints"].fillna(0).astype(int)
    return out


pipes_costs_region_df = cost_table("Region", region_list_global)
pipes_costs_subregion_df = cost_table("SubRegion", subregion_list_global)

# Sparse-sample fallbacks. A region with too few datapoints is replaced by the global
# mean; a subregion with too few datapoints is replaced by its (possibly-now-global)
# region mean. The thresholds are inclusive — DataPoints < MIN_* triggers the fallback.
MIN_REGION_DATAPOINTS = 5
MIN_SUBREGION_DATAPOINTS = 3
global_mean_cost = ratios_with_cost["CostUSDPerKm"].mean()
print(f"global mean CostUSDPerKm (post-trim): {global_mean_cost:,.0f}")

sparse_region_mask = (
    pipes_costs_region_df["DataPoints"] < MIN_REGION_DATAPOINTS
) | pipes_costs_region_df["CostUSDPerKm"].isna()
if sparse_region_mask.any():
    print(
        f"replacing region mean with global for {sparse_region_mask.sum()} region(s) "
        f"with < {MIN_REGION_DATAPOINTS} datapoints: "
        f"{pipes_costs_region_df.index[sparse_region_mask].tolist()}"
    )
pipes_costs_region_df.loc[sparse_region_mask, "CostUSDPerKm"] = global_mean_cost

sparse_subregion_mask = (
    pipes_costs_subregion_df["DataPoints"] < MIN_SUBREGION_DATAPOINTS
) | pipes_costs_subregion_df["CostUSDPerKm"].isna()
if sparse_subregion_mask.any():
    print(
        f"replacing subregion mean with region mean for {sparse_subregion_mask.sum()} "
        f"subregion(s) with < {MIN_SUBREGION_DATAPOINTS} datapoints"
    )
for sr in pipes_costs_subregion_df.index[sparse_subregion_mask]:
    pipes_costs_subregion_df.loc[sr, "CostUSDPerKm"] = pipes_costs_region_df.loc[
        dict_subregion_region[sr], "CostUSDPerKm"
    ]

# write per-km tables to Excel in USD millions per km for readability
(pipes_costs_region_df.assign(CostUSDMillionsPerKm=lambda d: d["CostUSDPerKm"] / 1e6)
    .drop(columns="CostUSDPerKm")
    .sort_values("CostUSDMillionsPerKm", ascending=False)
    .to_excel(excel_writer, sheet_name="Cost per km by region"))
(pipes_costs_subregion_df.assign(CostUSDMillionsPerKm=lambda d: d["CostUSDPerKm"] / 1e6)
    .drop(columns="CostUSDPerKm")
    .sort_values("CostUSDMillionsPerKm", ascending=False)
    .to_excel(excel_writer, sheet_name="Cost per km by subregion"))

pipes_costs_subregion_df.sort_values("CostUSDPerKm", ascending=False)


In [ ]:
# Per-pipeline cost estimate, then capex pivot for the selected REGION_NAME / FUEL bucket.
# Default estimate: known-length × subregion-mean per-km cost.
# Overwrite: rows that already have BOTH a known length and a known per-km cost on the
# pipeline itself get the actual product.
country_ratios_with_capex = country_ratios_df_subset.copy()
country_ratios_with_capex["CostUSDEstimate"] = (
    country_ratios_with_capex["LengthKnownKmByCountry"]
    * country_ratios_with_capex["SubRegion"].map(pipes_costs_subregion_df["CostUSDPerKm"])
)
known_cost_mask = (
    country_ratios_with_capex["LengthKnownKmByCountry"].notna()
    & country_ratios_with_capex["CostUSDPerKm"].notna()
)
country_ratios_with_capex.loc[known_cost_mask, "CostUSDEstimate"] = (
    country_ratios_with_capex.loc[known_cost_mask, "LengthKnownKmByCountry"]
    * country_ratios_with_capex.loc[known_cost_mask, "CostUSDPerKm"]
)


def pivot_capex(group_cols, index) -> pd.DataFrame:
    """Sum CostUSDEstimate by Status × group_cols, reshape to wide form, in USD billions."""
    grouped = (
        country_ratios_with_capex.groupby([*group_cols, "Status"])["CostUSDEstimate"]
        .sum()
        .unstack("Status")
        .reindex(columns=STATUS_LIST)
        .reindex(index=index)
        .fillna(0)
    ) / 1e9
    grouped[IN_DEV_COL] = grouped[["proposed", "construction"]].sum(axis=1)
    return grouped[EXCEL_STATUS_LIST]


capex_by_country = pivot_capex(["Country"], country_list)
capex_by_country.index.name = "Country"

capex_by_region = pivot_capex(["Region", "SubRegion"], multiindex_region_subregion)
capex_by_region.index.names = ["Region", "Subregion"]

capex_by_country.loc["Total"] = capex_by_country.sum(axis=0).values
capex_by_region.loc["Total"] = capex_by_region.sum(axis=0).values

# drop countries with no capex in any status, then blank out zeros for display
capex_by_country = capex_by_country.loc[~(capex_by_country == 0).all(axis=1)]
capex_by_country = capex_by_country.replace(0, "")
capex_by_region = capex_by_region.replace(0, "")

capex_by_region.to_excel(excel_writer, sheet_name="Capex USD billions by region")
capex_by_country.to_excel(excel_writer, sheet_name="Capex USD billions by country")
capex_by_region


## save Excel

In [ ]:
excel_writer.close()
print(f"wrote {output_path}")


## landing-page stats

In [ ]:
fueled = pipes_df_touse.loc[pipes_df_touse["Fuel"].isin(FUEL_OPTIONS)]
print(f"{len(fueled):>6,d} {FUEL_LABEL} pipeline projects tracked")
print(f"{fueled['LengthMergedKm'].sum() / 1e6:>6.3f} M km tracked")


In [ ]:
in_dev = pipes_df_touse.loc[
    pipes_df_touse["Fuel"].isin(FUEL_OPTIONS)
    & pipes_df_touse["Status"].isin(["proposed", "construction"])
]
print(f"{len(in_dev):>6,d} {FUEL_LABEL} pipeline projects in development (proposed + construction)")
print(f"{in_dev['LengthMergedKm'].sum() / 1e3:>6.1f} K km in development")
